<a href="https://colab.research.google.com/github/sonky20/sonky/blob/master/3%EC%9D%BC%EC%B0%A8_%ED%9A%8C%EA%B7%80%EB%AA%A8%EB%8D%B8%EC%8B%A4%EC%8A%B52.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [30]:
#라이브러리, 함수 로딩
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.metrics import *
from sklearn.preprocessing import MinMaxScaler
import torch
from torch import nn
from torch.utils.data import DataLoader, TensorDataset
from torch.optim import Adam

#디바이스 지정
device = "cuda" if torch.cuda.is_available() else "cpu"

#함수 생성
##데이터 로더 선언 함수
def make_DataSet(x_train, x_val, y_train, y_val, batch_size=32):
  #데이터 텐서로 변환
  x_train_tensor = torch.tensor(x_train, dtype=torch.float32)
  y_train_tensor = torch.tensor(y_train, dtype=torch.float32).view(-1, 1)
  x_val_tensor = torch.tensor(x_val, dtype=torch.float32)
  y_val_tensor = torch.tensor(y_val, dtype=torch.float32).view(-1, 1)
  #TensorDataset 생성: 텐서 데이터 세트로 합치기
  train_dataset = TensorDataset(x_train_tensor, y_train_tensor)
  #DataLoader 생성
  train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
  return train_loader, x_val_tensor, y_val_tensor

##학습 함수
def train(dataloader, model, loss_fn, optimizer, device):
  size = len(dataloader.dataset) #전체 데이터 세트의 크기
  num_batches = len(dataloader) #배치의 개수
  tr_loss = 0
  model.train() #학습 모드로 설정
  for x,y in dataloader: #배치 단위 로딩
    x, y = x.to(device), y.to(device) #디바이스 지정
    #Feed Forward(오차 순전파)
    pred = model(x)
    loss = loss_fn(pred, y)
    tr_loss += loss
    #Bakpropagation(오차 역전파)
    loss.backward() #역전파를 통해 각 파라미터에 대한 오차 기울기 계산
    optimizer.step() #옵티마이저가 모델의 파라미터를 업데이트
    optimizer.zero_grad() #옵티마이저의 기울기값 초기화
  tr_loss /= num_batches #모든 배치에서의 loss 평균
  return tr_loss.item()

##검증 평가 함수
def evaluate(x_val_tensor, y_val_tensor, model, loss_fn, device):
  model.eval() #모델을 평가 모드로 설정
  with torch.no_grad(): #평가 과정에서 기울기를 계산하지 않도록 설정
    x,y = x_val_tensor.to(device), y_val_tensor.to(device)
    pred = model(x)
    eval_loss = loss_fn(pred, y).item() #에측값 predd와 목표값 y 사이의 오차 계산
  return eval_loss, pred

  ##학습 곡선 함수
  def dl_learning_curve(tr_loss_list, val_loss_list):
    epochs = list(range(1, len(tr_loss_list)+1))
    plt.plot(epochs, tr_loss_list, label='train_err', marker='.')
    plt.plot(epochs, val_loss_list, label='val_err', marker='.')
    plt.xlabel('epoch')
    plt.ylabel('loss')
    plt.legend()
    plt.grid()
    plt.show()



In [31]:
#데이터 로딩
path = 'https://bit.ly/ds_boston_csv'
data = pd.read_csv(path)
data.head()

,crim,zn,indus,chas,nox,rm,age,dis,rad,tax,ptratio,lstat,medv
0,0.00632,18.0,2.31,0,0.538,6.575,65.2,4.0900,1,296,15.3,4.98,24.0
1,0.02731,0.0,7.07,0,0.469,6.421,78.9,4.9671,2,242,17.8,9.14,21.6
2,0.02729,0.0,7.07,0,0.469,7.185,61.1,4.9671,2,242,17.8,4.03,34.7
3,0.03237,0.0,2.18,0,0.458,6.998,45.8,6.0622,3,222,18.7,2.94,33.4
4,0.06905,0.0,2.18,0,0.458,7.147,54.2,6.0622,3,222,18.7,5.33,36.2


In [32]:
#x,y 분할
target = 'medv'
x = data.drop(target, axis=1)
y = data.loc[:, target]

#train, val 분할
x_train, x_val, y_train, y_val = train_test_split(x, y, test_size=0.2, random_state=20)

#스케일링
scaler = MinMaxScaler()
x_train = scaler.fit_transform(x_train)
x_val = scaler.transform(x_val)
print(x_train.dtype, x_val.dtype)

#y_train, y_val을 넘파이 배열로 변환
y_train = y_train.values
y_val = y_val.values
print(y_train.dtype, y_val.dtype)

#데이터 로더 준비
train_loader, x_val_tensor, y_val_tensor = make_DataSet(x_train, x_val, y_train, y_val, 32)

float64 float64
float64 float64


In [33]:
n_feature = x.shape[1]
print(x.shape[1])
model2 = nn.Sequential(nn.Linear(n_feature, 2), nn.ReLU(), nn.Linear(2,1), ).to(device) #은닉층 모델

12


In [34]:
#손실 함수, 옵티마이저 준비
loss_fn = nn.MSELoss()
optimizer = Adam(model2.parameters(), lr=0.01)

#학습
epochs = 100 #100번 반복 학습
tr_loss_list, val_loss_list = [], []
for epoch in range(epochs):
  tr_loss = train(train_loader, model2, loss_fn, optimizer, device)
  eval_loss, pred = evaluate(x_val_tensor, y_val_tensor, model2, loss_fn, device)
  tr_loss_list.append(tr_loss)
  val_loss_list.append(eval_loss)
  print(f'epoch: {epoch+1}, train_loss: {tr_loss:4f}, val_loss: {eval_loss:4f}')


epoch: 1, train_loss: 580.386536, val_loss: 501.049713
epoch: 2, train_loss: 572.435120, val_loss: 486.026276
epoch: 3, train_loss: 549.908752, val_loss: 455.994720
epoch: 4, train_loss: 510.866821, val_loss: 414.042877
epoch: 5, train_loss: 456.121307, val_loss: 362.814301
epoch: 6, train_loss: 397.114380, val_loss: 306.811981
epoch: 7, train_loss: 340.754364, val_loss: 251.974106
epoch: 8, train_loss: 284.433533, val_loss: 203.718384
epoch: 9, train_loss: 235.373474, val_loss: 166.301895
epoch: 10, train_loss: 197.654541, val_loss: 141.804367
epoch: 11, train_loss: 174.499649, val_loss: 127.613434
epoch: 12, train_loss: 160.371582, val_loss: 118.862991
epoch: 13, train_loss: 150.917343, val_loss: 113.042336
epoch: 14, train_loss: 147.738495, val_loss: 108.035034
epoch: 15, train_loss: 137.191589, val_loss: 103.236847
epoch: 16, train_loss: 131.037506, val_loss: 98.421730
epoch: 17, train_loss: 124.575729, val_loss: 93.731789
epoch: 18, train_loss: 120.662804, val_loss: 89.261635
epoc

In [40]:
#예측 결과
_, pred = evaluate(x_val_tensor, y_val_tensor, model2, loss_fn, device)
#모델 평가
print(f'MAE: {mean_absolute_error(y_val_tensor, pred):.2f}')
print(f'MSE: {mean_squared_error(y_val_tensor, pred):.2f}')
#print(f'RMSE: {mean_squared_error(y_val, pred, squared=False):.2f}')


MAE: 3.84
MSE: 24.12


In [38]:
x_train.shape

(404, 12)